# Create Amazon_Sale_Report SQLite Database

This notebook creates the SQLite database used by the SQL analysis from the cleaned CSV dataset.

In [ ]:
import pandas as pd
import sqlite3
from pathlib import Path

base_path = Path("..")
csv_path = base_path / "data" / "cleaned" / "Amazon_Sale_Report_Cleaned.csv"
db_path = base_path / "data" / "cleaned" / "Amazon_Sale_Report.db"

df = pd.read_csv(csv_path, sep=";")

print("CSV loaded successfully.")
print("Rows:", len(df))
print("Columns:", len(df.columns))

In [ ]:
# Prepare data types for SQLite
df["date"] = pd.to_datetime(df["date"], format="%d.%m.%Y", errors="coerce")

for column in ["amount"]:
    df[column] = pd.to_numeric(df[column], errors="coerce")

for column in ["qty", "ship_postal_code", "year", "month", "quarter"]:
    df[column] = pd.to_numeric(df[column], errors="coerce").astype("Int64")

boolean_columns = ["b2b", "is_cancelled", "is_returned", "has_promotion"]
for column in boolean_columns:
    df[column] = df[column].map({True: 1, False: 0, 1: 1, 0: 0}).astype("Int64")

print("Data types prepared successfully.")

In [ ]:
# Create / replace SQLite database
if db_path.exists():
    db_path.unlink()

with sqlite3.connect(db_path) as conn:
    df.to_sql("amazon_sales", conn, if_exists="replace", index=False)

print("SQLite database created successfully:")
print(db_path)

In [ ]:
# Validate database
with sqlite3.connect(db_path) as conn:
    row_count = pd.read_sql_query("SELECT COUNT(*) AS total_rows FROM amazon_sales", conn)
    unique_orders = pd.read_sql_query("SELECT COUNT(DISTINCT order_id) AS unique_orders FROM amazon_sales", conn)
    total_sales = pd.read_sql_query("SELECT ROUND(SUM(amount), 2) AS total_sales FROM amazon_sales", conn)

print("Rows:", int(row_count.iloc[0]["total_rows"]))
print("Unique orders:", int(unique_orders.iloc[0]["unique_orders"]))
print("Total sales amount:", total_sales.iloc[0]["total_sales"])